# Construcción del dataset — TA IA (1INF24), Grupo 1

Genera `dataset_distritos_v3.csv` (1 distrito = 1 fila) a partir de:
- Base de emergencias del SINPAD-INDECI (subir el Excel al panel de archivos de Colab)
- GeoJSON de distritos del INEI (se descarga desde GitHub)
- Altitud de la capital distrital del INEI (se descarga desde GitHub)

Ejecutar las celdas en orden.

### Celda 1 — Librerías y fuentes

In [1]:
import json
import numpy as np
import pandas as pd
import requests

ARCHIVO_INDECI = "BD-EMER-Y-DAÑOS-INTEGRADA-2003-2020-validada.xlsx"
URL_GEOJSON = "https://raw.githubusercontent.com/juaneladio/peru-geojson/master/peru_distrital_simple.geojson"
URL_ALTITUD = "https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv"

### Celda 2 — Leer la base del INDECI (una fila = una emergencia)

In [4]:
df = pd.read_excel(ARCHIVO_INDECI, sheet_name="BD 2003-2020")
df = df.dropna(subset=["COD. DISTRITO"])          # la última fila del Excel está vacía
df["ubigeo"] = (pd.to_numeric(df["COD. DISTRITO"], errors="coerce")
                  .astype("Int64").astype(str).str.zfill(6))
COL_FECHA = [c for c in df.columns if "FECHA" in c.upper()][0]
print("Emergencias:", len(df))

FileNotFoundError: [Errno 2] No such file or directory: 'BD-EMER-Y-DAÑOS-INTEGRADA-2003-2020-validada.xlsx'

### Celda 3 — Filtrar huaicos y diagnosticar duplicados

In [5]:
h = df[df["EMERGENCIA"].astype(str).str.upper().str.contains("HUAICO|HUAYCO", na=False)].copy()
DANOS = ["FALLECIDOS", "DAMNIFICADOS", "VIVIENDAS DESTRUIDAS",
         "VIVIENDAS AFECTADAS", "HAS CULTIVO DESTRUIDO"]
h[DANOS] = h[DANOS].apply(pd.to_numeric, errors="coerce").fillna(0)
print("Registros de huaico:", len(h))

clave = ["ubigeo", COL_FECHA]
rep = h[h.duplicated(clave, keep=False)]
print("Registros que comparten distrito y fecha:", len(rep))
print("  - copias exactas (mismos daños):", h.duplicated(clave + DANOS).sum())
print("  - grupos con daños distintos:",
      (rep.groupby(clave)[DANOS].nunique().max(axis=1) > 1).sum())

NameError: name 'df' is not defined

### Celda 4 — Eliminar solo las copias exactas

In [6]:
# Mismo distrito, fecha y daños = mismo reporte cargado dos veces.
# Mismo distrito y fecha con daños distintos = se conservan (pueden ser dos huaicos).
h = h.drop_duplicates(clave + DANOS, keep="first")
print("Registros tras eliminar copias exactas:", len(h))

NameError: name 'h' is not defined

### Celda 5 — Una fila por distrito y por período

In [ ]:
# huaicos = número de fechas distintas con huaico (no número de filas)
def agregar(sub, suf):
    g = sub.groupby("ubigeo").agg(
        huaicos=(COL_FECHA, "nunique"),
        fallecidos=("FALLECIDOS", "sum"),
        damnificados=("DAMNIFICADOS", "sum"),
        viv_destruidas=("VIVIENDAS DESTRUIDAS", "sum"),
        viv_afectadas=("VIVIENDAS AFECTADAS", "sum"),
        has_destruidas=("HAS CULTIVO DESTRUIDO", "sum"))
    return g.add_suffix(suf)

X = agregar(h[h["AÑO"] <= 2014], "_2003_2014")   # entradas del modelo
Y = agregar(h[h["AÑO"] >= 2015], "_2015_2020")[  # solo para construir la etiqueta
    ["huaicos_2015_2020", "damnificados_2015_2020",
     "fallecidos_2015_2020", "viv_destruidas_2015_2020"]]

### Celda 6 — Unir con los 1 834 distritos del GeoJSON (INEI)

In [ ]:
geo = requests.get(URL_GEOJSON).json()
d = pd.DataFrame([f["properties"] for f in geo["features"]])[["IDDIST", "NOMBDEP", "NOMBPROV", "NOMBDIST"]]
d.columns = ["ubigeo", "departamento", "provincia", "distrito"]
d["ubigeo"] = d["ubigeo"].astype(str).str.zfill(6)

faltan = sorted(set(h["ubigeo"]) - set(d["ubigeo"]))
print("Distritos con huaicos que no están en el GeoJSON:", len(faltan), faltan)

out = (d.merge(X, left_on="ubigeo", right_index=True, how="left")
        .merge(Y, left_on="ubigeo", right_index=True, how="left"))
num = [c for c in out.columns if c.endswith(("_2003_2014", "_2015_2020"))]
out[num] = out[num].fillna(0)
print("Filas:", len(out))

Distritos con huaicos que no están en el GeoJSON: 9 ['050409', '050509', '080913', '090511', '090720', '090722', '100113', '100607', '100610']
Filas: 1834


### Celda 7 — Índice de daño y etiqueta (definición principal)

In [ ]:
# El índice solo usa medidas de daño. Pesos 0,45/0,225/0,225 reescalados para sumar 1.
def z(c):
    m = out[c].max()
    return np.log1p(out[c]) / np.log1p(m) if m > 0 else 0 * out[c]

out["indice_dano"] = (0.50 * z("damnificados_2015_2020")
                      + 0.25 * z("fallecidos_2015_2020")
                      + 0.25 * z("viv_destruidas_2015_2020"))
con_dano = out["indice_dano"] > 0
corte = out.loc[con_dano, "indice_dano"].quantile(0.50)        # mediana
out["nivel_dano"] = np.where(~con_dano, "Bajo",
                     np.where(out["indice_dano"] >= corte, "Alto", "Medio"))

con_huaicos = out["huaicos_2015_2020"] > 0
print("Distritos con huaicos 2015-2020:", con_huaicos.sum(),
      "| sin ningún daño:", (con_huaicos & ~con_dano).sum())
print("Corte (mediana):", round(corte, 4))
print(out["nivel_dano"].value_counts())

Distritos con huaicos 2015-2020: 585 | sin ningún daño: 377
Corte (mediana): 0.2503
nivel_dano
Bajo     1626
Medio     104
Alto      104
Name: count, dtype: int64


### Celda 8 — Etiqueta alternativa (solo para el análisis de sensibilidad)

In [ ]:
# Incluye el número de huaicos en el índice (definición del entregable parcial).
indice_alt = (0.10 * z("huaicos_2015_2020") + 0.45 * z("damnificados_2015_2020")
              + 0.225 * z("fallecidos_2015_2020") + 0.225 * z("viv_destruidas_2015_2020"))
corte_alt = indice_alt[con_huaicos].quantile(0.75)
out["nivel_dano_alt"] = np.where(~con_huaicos, "Bajo",
                         np.where(indice_alt >= corte_alt, "Alto", "Medio"))
print(out["nivel_dano_alt"].value_counts())

nivel_dano_alt
Bajo     1249
Medio     438
Alto      147
Name: count, dtype: int64


### Celda 9 — Región natural (moda de las emergencias del distrito)

In [ ]:
reg = (df.dropna(subset=["REGIÓN NATURAL"])
         .groupby("ubigeo")["REGIÓN NATURAL"].agg(lambda s: s.mode().iloc[0]))
out["region_natural"] = out["ubigeo"].map(reg)
sin_region = out["region_natural"].isna()
print("Sin región en el INDECI:")
print(out.loc[sin_region, ["ubigeo", "departamento", "distrito"]])

mapa = {"AMAZONAS": "SELVA ALTA", "ANCASH": "SIERRA", "APURIMAC": "SIERRA", "AREQUIPA": "SIERRA",
        "AYACUCHO": "SIERRA", "CAJAMARCA": "SIERRA", "CALLAO": "COSTA", "CUSCO": "SIERRA",
        "HUANCAVELICA": "SIERRA", "HUANUCO": "SIERRA", "ICA": "COSTA", "JUNIN": "SIERRA",
        "LA LIBERTAD": "COSTA", "LAMBAYEQUE": "COSTA", "LIMA": "COSTA", "LORETO": "SELVA BAJA",
        "MADRE DE DIOS": "SELVA BAJA", "MOQUEGUA": "SIERRA", "PASCO": "SIERRA", "PIURA": "COSTA",
        "PUNO": "SIERRA", "SAN MARTIN": "SELVA ALTA", "TACNA": "SIERRA", "TUMBES": "COSTA",
        "UCAYALI": "SELVA BAJA"}
out.loc[sin_region, "region_natural"] = out.loc[sin_region, "departamento"].map(mapa)
out["region_asignada_por_dpto"] = sin_region.astype(int)
print(out["region_natural"].value_counts())

Sin región en el INDECI:
      ubigeo departamento                 distrito
778   150127         LIMA              PUNTA NEGRA
1686  010309     AMAZONAS               SAN CARLOS
1831  160109       LORETO                 PUTUMAYO
1833  160114       LORETO  TENIENTE MANUEL CLAVERO
region_natural
SIERRA        1182
COSTA          364
SELVA ALTA     144
SELVA BAJA     144
Name: count, dtype: int64


### Celda 10 — Altitud de la capital distrital (INEI)

In [ ]:
alt = pd.read_csv(URL_ALTITUD, dtype={"inei": str})[["inei", "altitude"]]
alt.columns = ["ubigeo", "altitud"]
out = out.merge(alt, on="ubigeo", how="left")
out["altitud_imputada"] = out["altitud"].isna().astype(int)
print("Sin altitud en la tabla del INEI:")
print(out.loc[out["altitud_imputada"] == 1, ["ubigeo", "provincia", "distrito"]])
out["altitud"] = out["altitud"].fillna(out.groupby("provincia")["altitud"].transform("median"))
print(out["altitud"].describe())

Sin altitud en la tabla del INEI:
      ubigeo provincia                 distrito
1831  160109    MAYNAS                 PUTUMAYO
1833  160114    MAYNAS  TENIENTE MANUEL CLAVERO
count    1834.000000
mean     2248.488550
std      1384.273176
min         5.000000
25%       678.250000
50%      2782.500000
75%      3374.000000
max      4705.000000
Name: altitud, dtype: float64


### Celda 11 — Verificaciones y exportación

In [ ]:
assert len(out) == 1834 and out["ubigeo"].is_unique
assert out.isna().sum().sum() == 0

orden = (["ubigeo", "departamento", "provincia", "distrito",
          "region_natural", "region_asignada_por_dpto", "altitud", "altitud_imputada"]
         + [c for c in out.columns if c.endswith("_2003_2014")]
         + [c for c in out.columns if c.endswith("_2015_2020")]
         + ["indice_dano", "nivel_dano", "nivel_dano_alt"])
out = out[orden]
out.to_csv("dataset_distritos_v3.csv", index=False, encoding="utf-8-sig")
print("Guardado: dataset_distritos_v3.csv", out.shape)

Guardado: dataset_distritos_v3.csv (1834, 21)
